# Build a Knowledge Graph for Your FHRP Gateways — the guided version

*A warm, step-by-step lab for network engineers. You bring the gateways. We bring the graph.*

**FHRP is the quiet protocol that decides which router is your default gateway.** HSRP,
VRRP, and GLBP all do the same trick: they put **one virtual IP on two routers** — one
**active**, one **standby** — so that when the active router dies, the standby picks up the
gateway in a few seconds (sub-second with tuned timers) and *the host never notices*. Every
laptop, till, camera, and
server points at that one virtual IP and quietly assumes someone is always home.

That "quietly assumes" is exactly where the danger hides. FHRP is so good at masking a
failure that a lost *backup* looks identical to a healthy network — right up until the
*second* router fails and a whole subnet loses its way out. A **knowledge graph** makes the
difference explicit: it can tell you, at 3 AM, whether the pager that just fired means
*"you lost your spare tyre"* or *"the subnet is stranded and revenue has stopped."*

By the end of this notebook you will have built, from an empty page, a small graph that
answers the question no `show standby` command can:

> *"The HSRP active router just failed. Is anything actually **down**, or did I just lose my
> backup?"*

and then, when the standby fails too, watch it name the exact service — **`POS Checkout`** —
that just lost its gateway. Same graph will also catch the classic FHRP trap: the **STP root**
and the **FHRP active** router sitting on *different* boxes.

### First, calm the nerves: this is **not** machine learning
No training. No model. No embeddings. No AI guessing in the dark. A knowledge graph is
just **structured facts** (things, and the named links between them) plus **queries** that
walk those links. Everything is **deterministic and inspectable** — run it twice, get the
same answer, and you can point at every node that produced it. If you can read the output of
`show standby brief`, you can read every line in this lab.

### What you need
Nothing. This runs in **Google Colab** with zero setup, using plain **Python + networkx +
matplotlib** (both already installed in Colab). No database, no Docker, no credentials.
Press *Runtime -> Run all*, or run each cell in order with **Shift+Enter**.

## The whole vocabulary — five words, each one an FHRP thing you already know

Before any code, here is the *entire* mental model. Everything after this is these five
ideas, repeated. You don't have to learn what any of them *mean* — only which FHRP thing
each one maps onto.

| Graph word | Plain meaning | The FHRP thing it already is |
|---|---|---|
| **node** | a thing | a router, an HSRP group, a subnet, a service |
| **edge** | a named, directed link between two nodes | "this router is the active for that group" |
| **label** | a node's *type* | `Router`, `FHRPGroup`, `Subnet`, `Service` |
| **property** | a fact stored *on* a node or edge | `state='down'`, `priority=110`, `virtual_ip='10.10.10.1'` |
| **traversal** | walking edges to answer a question | which subnet loses its gateway if these routers fail |

That's it. Hold those five words and the rest is just your day job wearing a new hat.

One idea deserves a spotlight, because it is the whole trick: **a fact can live on an edge,
not just a node.** "The active router failed" is not really a property of the router (the box
may be fine — its uplink dropped) and it is not a property of the group. It is a property of
the *role the router is currently playing*: its **`ACTIVE_FOR`** relationship to the group is
`down`. FHRP already thinks this way — active/standby is a *state a router holds toward a
group*, not a permanent fact about the box. In a graph you write it down literally: the
failure lives **on the edge**. Watch for that moment — it is where a wiring diagram turns
into something you can *query*.

## Setup — one import, one empty graph

**Theory (10 seconds).** `networkx` is a tiny Python library for building graphs in memory.
We will use a **`DiGraph`** — a *directed* graph, where every edge has a direction, an arrow.
That matters here: `dist-a --ACTIVE_FOR--> HSRP group` ("dist-a is the active for this group")
is a different statement from the reverse. FHRP is full of directional truth — who forwards,
who stands by, which group hands a gateway to which subnet — so directed edges are the honest
choice.

Run the next cell. It imports the tools and hands you an empty graph to fill.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# Both libraries ship pre-installed in Colab — nothing to pip install.
# A DiGraph is a *directed* graph: every edge is an arrow, from one node to another.
G = nx.DiGraph()

print("Empty graph ready:", G)
print("Nodes:", G.number_of_nodes(), " Edges:", G.number_of_edges())

**Checkpoint.** You should see `Empty graph ready` and `Nodes: 0  Edges: 0`. That empty `G`
is your blank whiteboard. Every step from here just adds nodes and edges to it.

## Step 1 — Two routers, and the roles they hold

**Theory.** A gateway that can fail over needs (at least) two routers. In the classic design
they are your two **distribution switches**, `dist-a` and `dist-b`. One is configured to win
the election and be **active**; the other waits as **standby**.

Which one wins is decided by **priority** — higher takes the active role. We give `dist-a`
priority **110** and `dist-b` priority **100**, so `dist-a` is the intended active. Notice the
word *intended*: the `role` we store is the **design intent**, not the live state. The box can
be the "active" by role while its active *duty* is currently down — exactly the distinction a
routing table blurs and a graph keeps sharp.

We also quietly record one more fact on `dist-a`: `stp_root=True`. Park that for now — it is
the seed of a trap we'll spring near the end. In graph terms each router becomes a **node**
with a `Router` **label** and a few **properties**.

In [ ]:
# add_node(name, **properties). role is the *intended* role; priority decides who wins active.
G.add_node("dist-a", label="Router", role="active",  priority=110, stp_root=True)
G.add_node("dist-b", label="Router", role="standby", priority=100, stp_root=False)

for n, d in G.nodes(data=True):
    print(f'{n}: role={d["role"]}, priority={d["priority"]}, stp_root={d["stp_root"]}')

**Checkpoint.** Two routers print: `dist-a` (active, priority 110, stp_root True) and
`dist-b` (standby, priority 100, stp_root False). They're floating unconnected for now — the
group they share comes next.

## Step 2 — The FHRP group: one virtual IP on two boxes

**Theory.** Here is the heart of FHRP. The two routers agree to share a single **virtual IP**
— say `10.10.10.1` — and a virtual MAC. Every host on the subnet uses that virtual IP as its
default gateway and never learns (or cares) which physical router is answering. That shared
identity is a *thing* in its own right, so it earns its own **node**: an `FHRPGroup`.

We'll model an **HSRP** group, number **10**, owning `10.10.10.1`. (VRRP and GLBP differ in
the details — VRRP is the IETF standard, GLBP can load-share across multiple forwarders — but
the *graph shape* is identical: a group node with a virtual IP, wired to the routers that back
it. Everything you build here re-labels straight across to VRRP or GLBP.)

In [ ]:
G.add_node("HSRP grp 10", label="FHRPGroup", protocol="HSRP",
           group_id=10, virtual_ip="10.10.10.1")

print(G.number_of_nodes(), "nodes so far:", list(G.nodes))
print("Facts on the group:", G.nodes["HSRP grp 10"])

**Checkpoint.** Three nodes now: `dist-a`, `dist-b`, and `HSRP grp 10`, whose facts show
`protocol='HSRP'` and `virtual_ip='10.10.10.1'`. The actors are on stage. No edges yet — and
the edges are where the whole story lives.

## Step 3 — The roles as edges that carry state (this is the pivotal step)

**Theory.** Here is the idea the whole lab pivots on. The pager just fired: **`dist-a`, the
HSRP active, lost its uplink.** *Where* does that fact belong? Not on the router node (the box
is powered and pingable). Not on the group (the virtual IP is fine — the standby is answering
it). It belongs on the **relationship between the router and the group** — the role `dist-a`
plays *toward* that group. That role is now **down**, and `dist-b`'s standby role is carrying
the gateway.

So we add two **edges** and hang a `state` **property** on each:

- `dist-a --ACTIVE_FOR(down)--> HSRP grp 10` — the active duty has failed. This edge is the incident.
- `dist-b --STANDBY_FOR(up)--> HSRP grp 10` — the standby is up and, right now, forwarding.

Read the first one literally: *"dist-a is the active for this group, and that role is down."*
This is the exact moment a wiring diagram becomes queryable: the failure is a fact you can
point at, sitting on one edge.

In [ ]:
# add_edge(source, target, **properties). The failover state is a property ON the edge.
G.add_edge("dist-a", "HSRP grp 10", rel="ACTIVE_FOR",  state="down")   # <-- the incident
G.add_edge("dist-b", "HSRP grp 10", rel="STANDBY_FOR", state="up")

for u, v, d in G.edges(data=True):
    print(f'{u} --{d["rel"]}({d["state"]})--> {v}')

**Checkpoint.** Two edges print, and exactly one reads `down`: the
`dist-a --ACTIVE_FOR(down)--> HSRP grp 10` line. That single edge, with a property on it, is
the 3 AM event — the HSRP active dropping — sitting in your graph as a fact you can now query
against. Note what the graph is *not* yet saying: whether anyone is actually cut off. Hold that
thought.

## See it — draw the graph

**Theory.** Text is great during an incident, but to *understand* a design you want the
picture. We'll colour nodes by their **label** (routers one colour, the group another) and draw
the one **down** edge as a dashed red arrow so the failure jumps out. This is the same instinct
as a topology diagram — except this diagram is generated *from the data*, so it can never drift
out of sync with the truth.

In [ ]:
def draw(G, title="FHRP knowledge graph"):
    colors = {"Router": "#3aa0ff", "FHRPGroup": "#0f7f74",
              "Subnet": "#9aa5ad", "Service": "#c8702a"}
    node_colors = [colors.get(G.nodes[n].get("label"), "#cccccc") for n in G]
    pos = nx.spring_layout(G, seed=7)   # fixed seed => stable, repeatable layout

    plt.figure(figsize=(10, 7))
    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=1900, alpha=0.92)
    nx.draw_networkx_labels(G, pos, font_size=9)

    down  = [(u, v) for u, v, d in G.edges(data=True) if d.get("state") == "down"]
    other = [(u, v) for u, v, d in G.edges(data=True) if (u, v) not in down]
    nx.draw_networkx_edges(G, pos, edgelist=other, arrows=True, arrowsize=16, edge_color="#8894a0")
    nx.draw_networkx_edges(G, pos, edgelist=down,  arrows=True, arrowsize=16,
                           edge_color="#d34b3f", width=2.0, style="dashed")
    nx.draw_networkx_edge_labels(G, pos, font_size=7,
        edge_labels={(u, v): d.get("rel", "") for u, v, d in G.edges(data=True)})

    plt.axis("off"); plt.title(title); plt.tight_layout(); plt.show()

draw(G)

**Reading the picture.** You should see the two routers (blue) and the group (teal) with role
arrows between them, and one **dashed red** arrow from `dist-a` to the group — the failed active
role. This is still just a topology of *devices*. It becomes a *knowledge* graph in the next two
steps, when we add the things a wiring diagram has never carried: the subnet that depends on this
gateway, and the business behind it.

## Step 4 — The subnet the group is gateway for

**Theory.** A gateway is only interesting because *something lives behind it*. The HSRP group
is the default gateway for a **subnet** — the VLAN of hosts that send every off-net packet to
`10.10.10.1`. So we add a `Subnet` node, `10.10.10.0/24` (VLAN 10), and wire the group to it
with a **`GATEWAY_FOR`** edge: *"this group is the gateway for that subnet."*

Notice this edge has **no `state`** — it is structural, part of the design, not a thing that
flaps. What flaps is the role edges from Step 3. Keeping "what's designed" and "what's currently
true" on different kinds of edge is a habit that pays off the moment you start querying.

In [ ]:
G.add_node("10.10.10.0/24", label="Subnet", vlan=10)
G.add_edge("HSRP grp 10", "10.10.10.0/24", rel="GATEWAY_FOR")

print("Subnet facts:", G.nodes["10.10.10.0/24"])
print("Edges into the subnet:")
for u, v, d in G.in_edges("10.10.10.0/24", data=True):
    print(f'  {u} --{d["rel"]}--> {v}')

**Checkpoint.** The subnet `10.10.10.0/24` now has one incoming edge:
`HSRP grp 10 --GATEWAY_FOR--> 10.10.10.0/24`. The gateway and the thing it serves are joined.
Still no *business* fact, though — that's the final, missing node, next.

## Step 5 — The business service: the node FHRP never had

**Theory.** This is the node your routers were never designed to hold, and the reason the whole
lab exists. HSRP knows priorities, virtual IPs, hello timers, active and standby. It has never
once known that `10.10.10.0/24` is where **POS Checkout** lives, and that if that subnet loses
its gateway, *the tills stop taking money*.

So we add a **`Service`** node, `POS Checkout`, with a `criticality` property, and wire it to the
subnet with a **`RUNS_ON`** edge: *"this service runs on that subnet."* One edge — and now a
first-hop-redundancy fact and a business fact are welded together in a place you can query. No
`show` command holds this link. It has never lived anywhere except tribal knowledge. You're about
to make it a first-class, walkable fact.

In [ ]:
G.add_node("POS Checkout", label="Service", criticality="critical")
G.add_edge("POS Checkout", "10.10.10.0/24", rel="RUNS_ON")

print("Graph now has", G.number_of_nodes(), "nodes and", G.number_of_edges(), "edges.")
print("The load-bearing link:", [f'{u} -RUNS_ON-> {v}'
      for u, v, d in G.edges(data=True) if d.get("rel") == "RUNS_ON"])
draw(G, title="FHRP knowledge graph — now with the business service")

**Checkpoint.** The graph has grown an orange `Service` node, `POS Checkout`, joined by a
`RUNS_ON` edge to the subnet. In the redrawn picture you can *trace with your finger*: the two
routers -> the group -> the subnet -> POS Checkout. In the next step we make the computer trace
it for you — and, crucially, decide when that trace actually means trouble.

## Step 6 — The 3 AM question: redundancy vs reachability

**Theory.** Everything so far was setup for this. A **traversal** is just walking edges to answer
a question. Ours has to be smarter than "is a router down?" — because with FHRP, *a down active
router usually means nothing is down at all.* The standby caught it. What you truly want to know
splits in two:

- **Lost the backup** — the active role is down, but a standby is still up. The gateway works;
  you've merely spent your redundancy. Reachable. *Fix it in the morning.*
- **Lost the gateway** — the group has **no** up gateway (active *and* standby both down, or there
  was never a standby). Now the subnet is stranded and its services are at risk. *Wake people up.*

So we write two small walkers. `gateway_health` classifies every group by counting its **up**
gateways: 2 = `REDUNDANT`, 1 = `NO BACKUP`, 0 = `NO GATEWAY`. Then `blast_radius` reports services
only for groups in `NO GATEWAY`. Run it against our current graph — where the active is down but
the standby is holding — and watch which bucket we land in.

In [ ]:
def gateway_health(G):
    '''For each FHRP group: how many of its routers can serve the gateway right now.'''
    report = []
    for grp, gd in G.nodes(data=True):
        if gd.get("label") != "FHRPGroup":
            continue
        # routers wired to this group as ACTIVE_FOR / STANDBY_FOR that are currently up
        up = [(r, ed["rel"]) for r, _, ed in G.in_edges(grp, data=True)
              if ed.get("rel") in ("ACTIVE_FOR", "STANDBY_FOR") and ed.get("state") == "up"]
        if len(up) == 0:
            status = "NO GATEWAY"     # active AND standby down -> subnet has no way out
        elif len(up) == 1:
            status = "NO BACKUP"      # one gateway left -> reachable, but redundancy is gone
        else:
            status = "REDUNDANT"      # active + standby both up
        report.append((grp, status, up))
    return report

def blast_radius(G):
    '''Services stranded when a group loses EVERY gateway (active and standby both down).'''
    at_risk = []
    for grp, status, up in gateway_health(G):
        if status != "NO GATEWAY":
            continue
        for _, sub, ed in G.out_edges(grp, data=True):
            if ed.get("rel") != "GATEWAY_FOR":
                continue
            for svc, _, ed2 in G.in_edges(sub, data=True):
                if ed2.get("rel") == "RUNS_ON":
                    at_risk.append((grp, sub, svc))
    return at_risk

print("Gateway health, group by group:")
for grp, status, up in gateway_health(G):
    holders = ", ".join(f"{r} ({rel.split('_')[0].lower()})" for r, rel in up) or "none"
    print(f"  {grp}: {status}   available: {holders}")

print()
hits = blast_radius(G)
for grp, sub, svc in hits:
    print(f"  NO GATEWAY on {grp} -> {sub} -> AT RISK: {svc}")
if not hits:
    print("  no services at risk — every subnet still has a working gateway")

Read what just happened. The pager screamed *"HSRP active on dist-a failed!"* — and your graph
answered: **`NO BACKUP`**, available gateway `dist-b (standby)`, and **no services at risk**. The
tills are still taking money. You didn't lose the gateway; you lost the *spare*. That is a
completely different night's sleep — and the router's log, which only ever said "state changed to
Active on dist-b," could never draw that line for you. The graph did, because you taught it the
one thing HSRP never knew: what runs behind the gateway, and how many gateways are left.

## Step 7 — What-if: now the standby fails too

**Theory.** Because each failover state lives on an edge as a plain property, you can *change* it
and re-ask — running "what if this one fails too?" on a **model**, safely, with no one paged and
no maintenance window. Right now we're at `NO BACKUP`: reachable, but one failure from the edge.
Let's take that failure. Flip `dist-b`'s standby role to `down` and the group crosses the line
from `NO BACKUP` to `NO GATEWAY` — and the same `blast_radius` query that returned *nothing* a
moment ago now names the service that just lost its way out.

In [ ]:
# You can read and write an edge's properties directly by [source][target].
print("active down, standby up :", blast_radius(G) or "nothing at risk (lost backup only)")

G["dist-b"]["HSRP grp 10"]["state"] = "down"      # the standby fails too
for grp, status, up in gateway_health(G):
    print(f"  {grp}: {status}")                    # watch it flip NO BACKUP -> NO GATEWAY
print("standby down as well    :", [svc for *_, svc in blast_radius(G)])

**Checkpoint.** The first line prints `nothing at risk (lost backup only)` — the standby was
covering. Then, after you knock the standby over too, the group's status flips to
`HSRP grp 10: NO GATEWAY` and the last line prints `['POS Checkout']`. Watch those two land
together: `NO GATEWAY` right beside the stranded service is the exact moment the whole lab is
built around. Same graph, same query — only the state on one edge changed, and the answer crossed
from *"lose sleep later"* to *"stranded now."* That is the redundancy-vs-reachability split made
mechanical: the model tells you which side of the line you're on. (We'll leave the group in this
failed state so the next exercise starts from a real outage.)

## Your turn #1 — a second service on the same subnet

Real subnets rarely carry just one thing. Suppose an **Inventory Sync** service also lives on
`10.10.10.0/24`. Add it, wire it with a `RUNS_ON` edge, and re-run `blast_radius`. Because the
group is currently in `NO GATEWAY` (we left both routers down), you should now see **two**
services surface from the same stranded subnet — one edge of extra truth is all it takes.

Fill in the cell below (there's a `# TODO`), then run it. The solution follows if you want to
check.

In [ ]:
# TODO: add an "Inventory Sync" Service node (criticality="high"),
#       then a RUNS_ON edge from "Inventory Sync" to "10.10.10.0/24".

# G.add_node(...)
# G.add_edge(...)

# The group already has no gateway (both routers down), so anything on this subnet is stranded:
print("services at risk:", sorted({svc for *_, svc in blast_radius(G)}))

<details><summary><b>Solution — click to reveal</b></summary>

```python
G.add_node("Inventory Sync", label="Service", criticality="high")
G.add_edge("Inventory Sync", "10.10.10.0/24", rel="RUNS_ON")

print("services at risk:", sorted({svc for *_, svc in blast_radius(G)}))
```

Now both `Inventory Sync` **and** `POS Checkout` come back from the one stranded subnet. The blast
radius grew the instant you told the graph one more true thing — you didn't touch the query at all.
</details>

In [ ]:
# (Solution, applied — then we repair both routers so the rest of the notebook starts healthy.)
G.add_node("Inventory Sync", label="Service", criticality="high")
G.add_edge("Inventory Sync", "10.10.10.0/24", rel="RUNS_ON")
print("Both routers down -> services at risk:", sorted({svc for *_, svc in blast_radius(G)}))

G["dist-a"]["HSRP grp 10"]["state"] = "up"        # active recovers
G["dist-b"]["HSRP grp 10"]["state"] = "up"        # standby recovers
print("Both routers repaired ->", [s for _, s, _ in gateway_health(G)])

## Quiet reveal — you have been writing an *ontology*

Time to name the thing you've been doing. Every step, you made two very different kinds of
decision, and it's worth seeing the seam between them:

- *"A `Router` holds an `ACTIVE_FOR` or `STANDBY_FOR` role toward an `FHRPGroup`. A group is
  `GATEWAY_FOR` a `Subnet`. A `Service` `RUNS_ON` a `Subnet`."* — these are the **rules**: which
  node types exist, which edges are allowed, what shape a valid fact takes. That is the **schema**.
  Its fancy name is an **ontology** — and here's the friendly definition: *an ontology is the RFC
  of your graph, the agreed vocabulary written down as structure.* You already trust RFC 2281 to
  say what a valid HSRP group is; an ontology does the same job for your graph.

- *"This particular router is `dist-a`, priority `110`, and its active role is `down`."* — these
  are the **instances**: the actual data.

A rule of thumb that keeps the two straight forever: **if it has a hostname, an IP, or a virtual
IP, it is data (an instance), never schema.** `Router` is schema; `dist-a` is data. `ACTIVE_FOR`
is schema; "dist-a is active for HSRP 10" is data. Keep the vocabulary small and stable; let the
instances be many. That single discipline is the difference between a graph you can grow for years
and a swamp you abandon in a month.

## A peek at the real thing (1/2) — the same group in Neo4j + Cypher

We used networkx so you could run everything with zero setup. But the *shapes* you built are
exactly what you'd build in a real graph database like **Neo4j**, whose query language is
**Cypher**. Cypher is worth a glance because it reads almost like the arrows we've been drawing —
`(node)-[:EDGE]->(node)`.

Here is **Steps 1–3 (the two routers and their roles)** as Cypher. `MERGE` means "find this node
or create it if missing" (so re-running is safe); `SET` assigns properties *after* the match — so
the failover state lands on the relationship without ever baking it into a `MERGE` pattern. This is
*reference only* — you don't run it here, it just shows the same idea in the grown-up tool:

```cypher
MERGE (a:Router {id: 'dist-a'})
SET   a.role = 'active', a.priority = 110, a.stp_root = true;

MERGE (b:Router {id: 'dist-b'})
SET   b.role = 'standby', b.priority = 100;

MERGE (grp:HSRPGroup {id: 'hsrp-10'})
SET   grp.virtual_ip = '10.10.10.1', grp.group_id = 10;

// the failover state, as a property on the relationship — same as our edge
MERGE (a)-[r_active:ACTIVE_FOR]->(grp)
SET   r_active.state = 'down';

MERGE (b)-[r_standby:STANDBY_FOR]->(grp)
SET   r_standby.state = 'up';
```

See it? `MERGE (a)-[r_active:ACTIVE_FOR]->(grp) SET r_active.state='down'` is the same statement as
our `G.add_edge("dist-a", "HSRP grp 10", rel="ACTIVE_FOR", state="down")`. Same node, same edge,
same fact-on-the-edge — one just happens to live in a database that scales to millions of nodes.

## A peek at the real thing (2/2) — the 3 AM question in Cypher

Our `blast_radius` walk is a Python function. In Cypher that same "no gateway left" traversal is a
few lines, because in a graph database you **draw the shape you're looking for** and let the engine
find every match — no manual loops:

```cypher
MATCH (grp:HSRPGroup)-[:GATEWAY_FOR]->(sub:Subnet)<-[:RUNS_ON]-(svc:BusinessService)
WHERE NOT EXISTS {
    MATCH (:Router)-[rel:ACTIVE_FOR|STANDBY_FOR]->(grp)
    WHERE rel.state = 'up'
}
RETURN grp.group_id AS group, sub.id AS subnet,
       collect(DISTINCT svc.name) AS services_at_risk;
```

Read the `WHERE NOT EXISTS { ... }` as a sentence: *"...and there is no router whose active-or-standby
role to this group is up."* That is exactly our `NO GATEWAY` test — a group with zero up gateways.
Swap the `NOT EXISTS` for a `count(...) = 1` and you'd be asking the `NO BACKUP` question instead:
same picture, one clause different. Different engine; identical thinking. If you can read the Python
above, you can read the Cypher — you already speak this language.

## Your turn #2 — the classic FHRP trap: root and active on different boxes

Now the payoff hiding in that `stp_root` property from Step 1. Two protocols are quietly deciding
your traffic's path at the distribution layer, and they must agree:

- **STP** decides which switch is the **root** — the top of the Layer 2 forwarding tree, the box
  every frame prefers to travel toward.
- **FHRP** decides which switch is the **active gateway** — the box that actually routes the subnet's
  traffic off-net.

If those land on **different** switches, every packet to the gateway takes an extra hop across the
inter-switch link to reach the active router — it *works*, so nobody notices, until that link
saturates or fails and you're debugging a "slow network" with no obvious cause. The rule is simple:
**the STP root and the FHRP active should sit on the same box.** Your graph stores both facts, so it
can just *ask*.

Below, `stp_fhrp_alignment` is written for you and run against the graph as built — where `dist-a` is
both the active *and* the STP root, so it reports nothing. **Your job:** in the TODO, move the STP root
onto `dist-b` (the standby) and re-run the check to watch the trap get flagged.

In [ ]:
def stp_fhrp_alignment(G):
    '''Flag any FHRP group whose active router is NOT the STP root switch.'''
    roots = [n for n, d in G.nodes(data=True) if d.get("stp_root")]
    findings = []
    for grp, gd in G.nodes(data=True):
        if gd.get("label") != "FHRPGroup":
            continue
        actives = [r for r, _, ed in G.in_edges(grp, data=True) if ed.get("rel") == "ACTIVE_FOR"]
        for active_sw in actives:
            for root_sw in roots:
                if active_sw != root_sw:
                    findings.append((grp, active_sw, root_sw))
    return findings

def report_alignment(G):
    findings = stp_fhrp_alignment(G)
    if not findings:
        print("aligned: STP root and FHRP active are on the same box for every group")
    for grp, active_sw, root_sw in findings:
        print(f"MISALIGNED {grp}: active gateway on {active_sw}, STP root on {root_sw}")

report_alignment(G)   # as built -> aligned (dist-a is both)

# TODO: move the STP root onto the standby, then re-run the check:
# G.nodes["dist-a"]["stp_root"] = False
# G.nodes["dist-b"]["stp_root"] = True
# report_alignment(G)

<details><summary><b>Solution — click to reveal</b></summary>

```python
G.nodes["dist-a"]["stp_root"] = False
G.nodes["dist-b"]["stp_root"] = True
report_alignment(G)
```

Now the check prints `MISALIGNED HSRP grp 10: active gateway on dist-a, STP root on dist-b`. You
changed no query and added no service — you moved one true fact, and the graph caught a design bug
that no single `show` command surfaces, because the bug lives in the *disagreement between two
protocols*, exactly the kind of cross-protocol truth a graph is built to hold.
</details>

In [ ]:
# (Solution, applied — then we put the root back on dist-a so later cells start aligned.)
G.nodes["dist-a"]["stp_root"] = False
G.nodes["dist-b"]["stp_root"] = True
report_alignment(G)

G.nodes["dist-a"]["stp_root"] = True     # realign the design
G.nodes["dist-b"]["stp_root"] = False
print("After realigning:")
report_alignment(G)

## Make it yours — swap in *your* gateways

Now the moment it becomes your tool, not mine. Change the values below to **one** active router,
**one** standby, **one** FHRP group (and its virtual IP), **one** subnet, and **one** service
*you* actually run. We model the worst case for you — both routers down — so your service shows
up. Run it, and
watch **your own service name** come back from `blast_radius`, proof that the machine you built
understands your network, not just the demo's.

Keep it to the smallest slice that answers one real question. You can always add one more node
tomorrow — that's the whole promise of a graph you can grow.

In [ ]:
# --- change these five values to your network ---
MY_ACTIVE  = "dc-core-a"
MY_STANDBY = "dc-core-b"
MY_GROUP   = "VRRP 20 (10.20.0.1)"
MY_VIP     = "10.20.0.1"
MY_SUBNET  = "10.20.0.0/24"
MY_SERVICE = "Badge Access"
# ------------------------------------------------

# Note: we keep the HSRP role names (active/standby, ACTIVE_FOR/STANDBY_FOR) here for
# consistency with the walkthrough; VRRP itself calls these roles master/backup.
G.add_node(MY_ACTIVE,  label="Router", role="active",  priority=110)
G.add_node(MY_STANDBY, label="Router", role="standby", priority=100)
G.add_node(MY_GROUP,   label="FHRPGroup", protocol="VRRP", virtual_ip=MY_VIP)
G.add_node(MY_SUBNET,  label="Subnet")
G.add_node(MY_SERVICE, label="Service", criticality="critical")

G.add_edge(MY_ACTIVE,  MY_GROUP,  rel="ACTIVE_FOR",  state="down")   # active gateway failed
G.add_edge(MY_STANDBY, MY_GROUP,  rel="STANDBY_FOR", state="down")   # and no standby to cover
G.add_edge(MY_GROUP,   MY_SUBNET, rel="GATEWAY_FOR")
G.add_edge(MY_SERVICE, MY_SUBNET, rel="RUNS_ON")

print(f"If {MY_GROUP} loses every gateway, these services are at risk:")
for grp, sub, svc in blast_radius(G):
    if grp == MY_GROUP:
        print(f"  AT RISK: {svc}   (on {sub})")

**Checkpoint.** Your own service name — `Badge Access`, or whatever you typed — prints as *at
risk*. That is the moment the tool stopped being a tutorial and became yours. Every other gateway
you run is just five more lines away. (Set the standby's `state` to `"up"` instead and re-run —
your service drops off the list, because now the group is merely `NO BACKUP`, not `NO GATEWAY`.)

## The one rule that keeps this from becoming a swamp

You'll be tempted to model everything. Don't. Here is the discipline, in one line:

> **Model the nouns of your design review, not the numbers of your monitoring dashboard.**

Routers, roles, groups, virtual IPs, subnets, services, the STP root — the **nouns** you'd draw on
a whiteboard while arguing about a failover design — those belong in the graph. Hello/hold timers,
per-second preemption counters, interface load, the full ARP table — those are the **numbers**;
leave them in the systems that already store them well, and let the graph *reference* them when it
needs to. The graph holds the *shape of the dependency* — who backs whom, what rides on what; your
NMS holds the firehose. Keep that line sharp and your graph stays queryable for years.

### Where to go next
- **The companion Neo4j lab** does this exact thing in a real graph database — same nodes, same
  edges, same 3 AM question — so the two Cypher peeks above become things you run.
- **Add tracking.** Model the uplink an FHRP group *tracks*, as a node the group points at, and
  "if this WAN interface drops, which gateway preempts, and does anything lose its backup?" becomes
  one more traversal.
- **Add a protocol.** The shapes generalize: a VRRP or GLBP group is the same group node re-labelled;
  an STP instance is a node that `ELECTS` a root; a redistribution policy is a node other things point
  at. The same five words carry all of it.

## You built a knowledge graph

Look back at the distance. You started with an empty page and five plain words. You added two
routers and a failed active role — a topology. Then you added the subnet that depends on that
gateway and the one node FHRP never had, the **business service** — and the topology became
*knowledge*. Finally you asked it the question no router can answer — *"is this a lost backup or a
lost gateway?"* — and it told you **`NO BACKUP`, nothing at risk** when the standby was holding,
then named **`POS Checkout`** the instant the standby failed too. It even caught the STP-root /
FHRP-active trap, a bug that lives in the space *between* two protocols.

The important idea was never HSRP, and never networkx. It's this: **operational truth has a shape**
— two routers back a group, a group is the gateway for a subnet, services depend on that subnet —
and once that shape is a graph, "did we just lose redundancy or lose the gateway?" stops being a
judgement call at 3 AM and becomes a traversal. A runbook is a hammer. A graph you can query is the
whole toolbox.

Every host on your network is quietly trusting that one virtual IP. Now you know how to build the
graph that remembers what happens when the routers behind it run out. Go model one real gateway on
your network, and ask it what breaks.